In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
results = pd.read_csv('sobel_results/detection_results.csv')

In [ ]:
def filter_rapid_changes(results, window_size=10, rapid_change_threshold_px=10, inplace=False):
    local_median = results['balise_height_px'].rolling(window=window_size, center=True, min_periods=1).median()
    filtered_results = results.copy() if not inplace else results
    filtered_results = filtered_results[filtered_results['balise_height_px'].sub(local_median).abs().le(rapid_change_threshold_px)]
    return filtered_results


def filter_min_max_height(results, min_height=10, max_height=67, inplace=False):
    filtered_results = results.copy() if not inplace else results
    filtered_results = filtered_results[
        (filtered_results['balise_height_px'] > min_height) &
        (filtered_results['balise_height_px'] < max_height)
    ]
    return filtered_results


def apply_filters(results, inplace=False):
    filtered_results = results.dropna(subset=['creation_date'])
    filtered_results = filter_min_max_height(filtered_results, inplace=inplace)
    filtered_results = filter_rapid_changes(filtered_results, inplace=inplace)
    return filtered_results


def add_moving_average(results, rolling_window='24h', inplace=False):
    """
    Add a moving average column to the results DataFrame.
    Parameters:
    - results: DataFrame containing the detection results with 'creation_date' and 'balise_height_px' columns.
    - rolling_window: The window size for the moving average (e.g., '24h' for 24 hours, or an integer for a fixed number of rows).
    - inplace: If True, modify the original DataFrame; if False, return a new DataFrame with the moving average.
    Returns:
    - A DataFrame with an added 'balise_height_px_moving_average' column containing the moving average of 'balise_height_px'.
    """
    
    average_results = results.copy() if not inplace else results
    average_results['creation_date'] = pd.to_datetime(average_results['creation_date'], errors='coerce')
    average_results['balise_height_px_moving_average'] = average_results.set_index('creation_date')['balise_height_px'].rolling(window=rolling_window, min_periods=1).mean().to_numpy()

    return average_results


def convert_px_to_metres(results, px_per_metre, inplace=False):
    """
    Convert the balise height from pixels to metres using a known scale factor.

    The scale factor (px_per_metre) must be calibrated from a reference object
    of known real-world size visible in the images.

    Parameters:
    - results: DataFrame containing 'balise_height_px' (and optionally
               'balise_height_px_moving_average') columns.
    - px_per_metre: float — number of pixels per metre, determined by calibration.
    - inplace: If True, modify the original DataFrame; if False, return a new copy.

    Returns:
    - A DataFrame with added 'balise_height_m' and, when present,
      'balise_height_m_moving_average' columns.
    """
    converted = results.copy() if not inplace else results
    converted['balise_height_m'] = converted['balise_height_px'] / px_per_metre
    if 'balise_height_px_moving_average' in converted.columns:
        converted['balise_height_m_moving_average'] = converted['balise_height_px_moving_average'] / px_per_metre
    return converted


In [ ]:
results

In [ ]:
results['creation_date'] = pd.to_datetime(results['creation_date'], errors='coerce')

In [ ]:
filtered_results = apply_filters(results, inplace=False)
filtered_results = add_moving_average(filtered_results)

ax = results.plot(
    x='creation_date',
    y='balise_height_px',
    figsize=(12, 6),
    alpha=0.35,
    linewidth=1,
    label='Hauteur détectée',
)

filtered_results.plot(
    x='creation_date',
    y='balise_height_px_moving_average',
    ax=ax,
    linewidth=2.5,
    label=f'Hauteur filtrée (moyenne mobile {24}h + filtrage)',
)
ax.invert_yaxis()
ax.set_title('Hauteur de la balise au fil du temps')
ax.set_xlabel('Date')
ax.set_ylabel('Hauteur de la balise (px)')
ax.grid(True)
ax.legend(loc='lower right')
plt.tight_layout()

In [ ]:
# --- Pixel-to-metre calibration -------------------------------------------
# En prenant celle avec le moins de neige, 2024-09-03 (RCNX0486.JPG), si on se dit que la balise est totalement sortie, alors 62px = 40cm et c’est une règle de 3.

px_per_metre = 155  # TODO: replace with the calibrated scale factor

results_m = convert_px_to_metres(
    add_moving_average(apply_filters(results)),
    px_per_metre=px_per_metre,
)

results_m['snow_level_m'] =  results_m['balise_height_m_moving_average'].max() - results_m['balise_height_m_moving_average']

ax = results_m.plot(
    x='creation_date',
    y='snow_level_m',
    linewidth=2.5,
    label='Niveau de neige',
)
# ax.invert_yaxis()
ax.set_title('Niveau de neige estimé au fil du temps')
ax.set_xlabel('Date')
ax.set_ylabel('Niveau de neige (m)')
ax.grid(True)
plt.tight_layout()

In [ ]:
meteo = pd.read_csv('open-meteo-45.87N6.98E3946m.csv', skiprows=3)
meteo['time'] = pd.to_datetime(meteo['time'], errors='coerce')
meteo.head()

In [ ]:
ax = results_m.plot(
    x='creation_date',
    y='snow_level_m',
    linewidth=2.5,
    label='Niveau de neige',
)

ax2 = ax.twinx()
# meteo.plot(
#     x='time',
#     y='snowfall_sum (cm)',
#     ax=ax2,
#     color='orange',
#     label='Précipitations de neige (cm)'
# )
# ax2.legend('lower left')

meteo.plot(
    x='time',
    y='precipitation_sum (mm)',
    ax=ax2,
    color='red',
    label='Précipitations totales (mm)'
)
ax2.legend(loc='upper right')

ax.set_title('Niveau de neige estimé au fil du temps')
ax.set_xlabel('Date')
ax.set_ylabel('Niveau de neige (m)')
ax.grid(True)
plt.tight_layout()

In [ ]:
meteo[50:]